# 04 — Integração, Correlações e Cálculo Alpha do IPB

**Objetivo**: cruzar todos os pilares, calcular correlações e gerar uma primeira versão do IPB (alpha).

**Inputs**: `data/processed/trusted_municipios_eda.parquet`.

**Outputs**:
- `data/processed/ipb_alpha.parquet`
- `data/processed/reports/ipb_alpha_report.json`
- Figuras de correlação e ranking.

In [ ]:
# Imports
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import PROCESSED_DATA_DIR
from src.utils.eda import (
    min_max_normalize,
    plot_correlation_heatmap,
    plot_distribution,
    save_figure,
    save_json,
    save_parquet,
    winsorize_series,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Leitura
df = pd.read_parquet(PROCESSED_DATA_DIR / "trusted_municipios_eda.parquet")
logger.info("Linhas: %d, Colunas: %d", df.shape[0], df.shape[1])

## 1. Seleção de variáveis por pilar

In [ ]:
variaveis_ipb = {
    "A_capacidade_consumo": ["rendimento_domiciliar_per_capita", "pib_per_capita"],
    "B_dinamismo": ["pix_per_capita_12m"],
    "C_adocao_digital": ["banda_larga_fixa_por_100_hab"],
    "D_gap_bancario": [
        "agencias_por_100k_hab",
        "depositos_per_capita",
        "credito_per_capita",
    ],
    "E_perfil_demografico": [
        "populacao_18_35_pct",
        "populacao_urbana_pct",
        "escolaridade_ensino_medio_pct",
    ],
}

all_vars = [v for lst in variaveis_ipb.values() for v in lst]
logger.info("Variáveis selecionadas: %s", all_vars)

## 2. Winsorização e normalização

In [ ]:
df_ipb = df[
    [
        "id_municipio",
        "nome_municipio",
        "sigla_uf",
        "nome_regiao",
        "populacao_total",
        "estrato_populacional",
    ]
].copy()

for pilar, vars_list in variaveis_ipb.items():
    for var in vars_list:
        col_norm = f"{var}_norm"
        df_ipb[col_norm] = min_max_normalize(
            winsorize_series(df[var].fillna(0), lower=0.01, upper=0.99)
        )

logger.info("Normalização concluída. Shape: %s", df_ipb.shape)

## 3. Inversão do pilar D (gap bancário)

In [ ]:
for var in variaveis_ipb["D_gap_bancario"]:
    df_ipb[f"{var}_norm"] = 1 - df_ipb[f"{var}_norm"]

logger.info("Pilar D invertido")

## 4. Cálculo dos pilares e do IPB alpha

In [ ]:
for pilar, vars_list in variaveis_ipb.items():
    cols = [f"{var}_norm" for var in vars_list]
    df_ipb[pilar] = df_ipb[cols].mean(axis=1)

# Média geométrica dos 5 pilares × 100
pilar_cols = list(variaveis_ipb.keys())
df_ipb["ipb_alpha"] = (df_ipb[pilar_cols].prod(axis=1)) ** (1 / 5) * 100
df_ipb["rank_ipb_alpha"] = (
    df_ipb["ipb_alpha"].rank(ascending=False, method="min").astype(int)
)

logger.info("IPB alpha - média: %.2f, mediana: %.2f", df_ipb["ipb_alpha"].mean(), df_ipb["ipb_alpha"].median())

## 5. Correlação entre pilares

In [ ]:
plot_correlation_heatmap(
    df_ipb,
    pilar_cols,
    method="spearman",
    title="Correlação entre Pilares",
    filename="04_correlacao_pilares.png",
)
plt.show()

## 6. Distribuição do IPB alpha

In [ ]:
plot_distribution(df_ipb, "ipb_alpha", filename="04_dist_ipb_alpha.png")
plt.show()

## 7. Top e bottom 30 do ranking

In [ ]:
top30 = df_ipb.nsmallest(30, "rank_ipb_alpha")[
    ["nome_municipio", "sigla_uf", "ipb_alpha", "rank_ipb_alpha"]
]
bottom30 = df_ipb.nlargest(30, "rank_ipb_alpha")[
    ["nome_municipio", "sigla_uf", "ipb_alpha", "rank_ipb_alpha"]
]

display(top30)
display(bottom30)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top30_sorted = top30.sort_values("ipb_alpha")
ax.barh(top30_sorted["nome_municipio"], top30_sorted["ipb_alpha"], color="steelblue")
ax.set_title("Top 30 Municípios - IPB Alpha")
ax.set_xlabel("IPB Alpha")
save_figure(fig, "04_top30_ipb_alpha.png")
plt.show()

## 8. PCA (exploratório)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df_ipb[pilar_cols].fillna(0)
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

df_ipb["pca1"] = components[:, 0]
df_ipb["pca2"] = components[:, 1]

fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df_ipb, x="pca1", y="pca2", hue="nome_regiao", alpha=0.6, ax=ax)
ax.set_title(f"PCA dos Pilares (var. explicada: {pca.explained_variance_ratio_.sum():.1%})")
save_figure(fig, "04_pca_pilares.png")
plt.show()

logger.info("Variância explicada pelos 2 componentes: %.2f%%", pca.explained_variance_ratio_.sum() * 100)

## 9. Salvamento

In [ ]:
save_parquet(df_ipb, "ipb_alpha.parquet")

ipb_report = {
    "media_ipb_alpha": float(df_ipb["ipb_alpha"].mean()),
    "mediana_ipb_alpha": float(df_ipb["ipb_alpha"].median()),
    "top_10": top30.head(10).to_dict(orient="records"),
    "bottom_10": bottom30.head(10).to_dict(orient="records"),
    "variancia_explicada_pca": float(pca.explained_variance_ratio_.sum()),
}
save_json(ipb_report, "ipb_alpha_report.json")